In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Municipality mapping
brussels_communes = {
    "Anderlecht": 21001,
    "Auderghem": 21002,
    "Berchem-Sainte-Agathe": 21003,
    "Ville de Bruxelles": 21004,
    "Etterbeek": 21005,
    "Evere": 21006,
    "Forest": 21007,
    "Ganshoren": 21008,
    "Ixelles": 21009,
    "Jette": 21010,
    "Koekelberg": 21011,
    "Molenbeek-Saint-Jean": 21012,
    "Saint-Gilles": 21013,
    "Saint-Josse-ten-Noode": 21014,
    "Schaerbeek": 21015,
    "Uccle": 21016,
    "Watermael-Boitsfort": 21017,
    "Woluwe-Saint-Lambert": 21018,
    "Woluwe-Saint-Pierre": 21019,
}

# Create reverse mapping (code -> name)
code_to_name = {v: k for k, v in brussels_communes.items()}

In [ ]:
workers = pd.read_csv("output/commuters_travelling.csv")
workers_all = pd.read_csv("output/commuters_with_addresses_new.csv")

# HOME <> WORK TRAVELS

In [ ]:
# check if the influx to Brussels municipalities is aligned with the Census 2021 data
census_2021_od_matrix = pd.read_excel(
    'input_data/working_population_home_work_municipality_2021.xlsx',
    sheet_name=0,
    header=None
)

In [ ]:
# Column 0 = NIS code, Column 1 = municipality name, Column 2 = total, Column 3 = total in Belgium, Column 4 = total in Brussels
# Column 5 - Column X = total in each municipality, 21001-21019 for Brussels municipalities

# Only keep the data from Flandres (NIS = 02000) and Wallonia (NIS = 03000)
census_2021_od_matrix = census_2021_od_matrix[census_2021_od_matrix[0].isin(['02000', '03000'])]

# Extract relevant columns
census_2021_od_matrix_filtered = census_2021_od_matrix[[0, 1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]].copy()
census_2021_od_matrix_filtered.columns = ['nis_code', 'region', 'total_in_brussels', '21001', '21002', '21003', '21004', '21005', '21006', '21007', '21008', '21009', '21010', '21011', '21012', '21013', '21014', '21015', '21016', '21017', '21018', '21019']


In [ ]:
# Reshape df_parsed from wide to long format
census_2021_long = census_2021_od_matrix_filtered.melt(
    id_vars=['nis_code', 'region', 'total_in_brussels'],
    var_name='brussels_municipality_code',
    value_name='number_of_people'
)

# Convert municipality code and number of people to numeric
census_2021_long['brussels_municipality_code'] = pd.to_numeric(census_2021_long['brussels_municipality_code'])
census_2021_long['number_of_people'] = pd.to_numeric(census_2021_long['number_of_people'])

# Add municipality name using the code_to_name mapping
census_2021_long['brussels_municipality_name'] = census_2021_long['brussels_municipality_code'].map(code_to_name)

In [ ]:
# Extract first 5 digits from assigned_work_sector to get municipality code
workers_all['work_municipality_code'] = workers_all['work_sector'].str[:5].astype(int)

# Group by work municipality code
workers_by_work_municipality = workers_all.groupby('work_municipality_code').size().to_frame('count')

# print(workers_by_work_municipality)

In [ ]:
# Split workers by home region
# Région flamande -- starts with 1, 3, 4, 7 or starts with 23/24 --> 02000
# Région wallonne -- otherwise --> 03000
home_muni = workers_all['municipality_nis'].astype('string')
print(home_muni)

is_flanders = (
    home_muni.str[:2].isin(['23', '24'])
    | home_muni.str[0].isin(['1', '3', '4', '7'])
)
workers_all['home_region'] = np.where(is_flanders.fillna(False), '02000', '03000')

# Group by work municipality code and home region
workers_by_work_muni_home_region = (
    workers_all
    .groupby(['home_region', 'work_municipality_code'])
    .size()
    .to_frame('count')
)

# print(workers_by_work_muni_home_region)

In [ ]:
# Merge workers data with census data
# First, sum census data by brussels municipality (across regions)
census_by_municipality = (
    census_2021_long
    .groupby('brussels_municipality_code')['number_of_people']
    .sum()
    .to_frame('census_count')
)

# Merge with workers data
comparison_df = workers_by_work_municipality.join(census_by_municipality, on='work_municipality_code')
comparison_df.columns = ['workers_count', 'census_count']

# Calculate difference and percentage difference
comparison_df['difference'] = comparison_df['workers_count'] - comparison_df['census_count']
comparison_df['pct_difference'] = (comparison_df['difference'] / comparison_df['census_count'] * 100).round(2)

# Add municipality names
comparison_df['municipality_name'] = comparison_df.index.map(code_to_name)

print(comparison_df[['municipality_name', 'workers_count', 'census_count', 'difference', 'pct_difference']])

In [ ]:
# Create a bar plot comparing workers count vs census count by municipality
fig, ax = plt.subplots(figsize=(14, 6))

municipalities = comparison_df['municipality_name'].values
x = np.arange(len(municipalities))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['workers_count'].values, width, label='Workers Count', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison_df['census_count'].values, width, label='Census Count', alpha=0.8)

ax.set_xlabel('Municipality')
ax.set_ylabel('Count')
ax.set_title('Workers Count vs Census Count by Brussels Municipality')
ax.set_xticks(x)
ax.set_xticklabels(municipalities, rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Merge workers data with census data at regional level (home region x Brussels work municipality)

# Workers counts: already grouped by ['home_region', 'work_municipality_code']
workers_region = workers_by_work_muni_home_region.reset_index().rename(columns={'count': 'workers_count'})

# Census counts by region code (02000/03000) and Brussels municipality code
census_region = (
    census_2021_long
    .rename(columns={'nis_code': 'home_region', 'brussels_municipality_code': 'work_municipality_code'})
    .groupby(['home_region', 'work_municipality_code'], as_index=False)['number_of_people']
    .sum()
    .rename(columns={'number_of_people': 'census_count'})
)

# Ensure compatible dtypes for merge keys
workers_region['home_region'] = workers_region['home_region'].astype(str)
census_region['home_region'] = census_region['home_region'].astype(str)
workers_region['work_municipality_code'] = pd.to_numeric(workers_region['work_municipality_code'], errors='coerce')
census_region['work_municipality_code'] = pd.to_numeric(census_region['work_municipality_code'], errors='coerce')

# Merge
comparison_df_region = workers_region.merge(
    census_region,
    on=['home_region', 'work_municipality_code'],
    how='left'
)

# Fill missing census counts with 0 where needed
comparison_df_region['census_count'] = comparison_df_region['census_count'].fillna(0)

# Calculate difference and percentage difference
comparison_df_region['difference'] = comparison_df_region['workers_count'] - comparison_df_region['census_count']
comparison_df_region['pct_difference'] = np.where(
    comparison_df_region['census_count'] > 0,
    (comparison_df_region['difference'] / comparison_df_region['census_count'] * 100).round(2),
    np.nan
)

# Add municipality names
comparison_df_region['municipality_name'] = comparison_df_region['work_municipality_code'].map(code_to_name)

print(
    comparison_df_region[
        ['home_region', 'work_municipality_code', 'municipality_name', 'workers_count', 'census_count', 'difference', 'pct_difference']
    ].sort_values(['home_region', 'work_municipality_code'])
)

In [ ]:
# Create separate bar plots for Flanders (02000) and Wallonia (03000)
plot_df = comparison_df_region.copy()
plot_df = plot_df.sort_values(['home_region', 'work_municipality_code'])

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
region_specs = [
    ('02000', 'Flanders'),
    ('03000', 'Wallonia')
]

for ax, (region_code, region_label) in zip(axes, region_specs):
    region_df = plot_df[plot_df['home_region'] == region_code].copy()
    region_df = region_df.sort_values('work_municipality_code')

    municipalities = region_df['municipality_name'].values
    x = np.arange(len(municipalities))
    width = 0.35

    ax.bar(
        x - width / 2,
        region_df['workers_count'].values,
        width,
        label='Workers Count',
        alpha=0.8
    )
    ax.bar(
        x + width / 2,
        region_df['census_count'].values,
        width,
        label='Census Count',
        alpha=0.8
    )

    ax.set_xlabel('Municipality')
    ax.set_title(f'{region_label} → Brussels: Workers vs Census')
    ax.set_xticks(x)
    ax.set_xticklabels(municipalities, rotation=45, ha='right')
    ax.legend()

axes[0].set_ylabel('Count')

plt.tight_layout()
plt.show()

# MODE SPLIT

In [ ]:
# Split of transport mode
mode_split = (
    workers["mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .to_frame("count")
)

mode_split["share_pct"] = (mode_split["count"] / mode_split["count"].sum() * 100).round(2)
print(mode_split)

In [ ]:
avg_distance_by_home_region = (
    workers
    .groupby("region", dropna=False)["commute_dist_km"]
    .mean()
    .sort_index()
    .to_frame("avg_commute_distance_km")
)

print(avg_distance_by_home_region)

# DEPARTURE TIMES

In [ ]:
home_to_work_min = workers["departure_home_to_work"]
work_to_home_min = pd.to_numeric(workers["departure_from_work"], errors="coerce").dropna()

bins = np.arange(0, 24 * 60 + 15, 15)  # 15-minute bins
xticks = np.arange(0, 24 * 60 + 1, 120)
xtick_labels = [f"{h:02d}:00" for h in range(0, 25, 2)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].hist(home_to_work_min, bins=bins, color="royalblue", alpha=0.8)
axes[0].set_title("Departure from Home to Work")
axes[0].set_xlabel("Time of day")
axes[0].set_ylabel("Number of trips")
axes[0].set_xticks(xticks)
axes[0].set_xticklabels(xtick_labels, rotation=45)

axes[1].hist(work_to_home_min, bins=bins, color="darkorange", alpha=0.8)
axes[1].set_title("Departure from Work to Home")
axes[1].set_xlabel("Time of day")
axes[1].set_xticks(xticks)
axes[1].set_xticklabels(xtick_labels, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Convert departure times to comparable format and check for agents with home-to-work > work-to-home
comparison = workers[['participant_id', 'departure_home_to_work', 'departure_from_work']].copy()
comparison['departure_from_work'] = pd.to_numeric(comparison['departure_from_work'], errors='coerce')

# Find cases where departure_home_to_work > departure_from_work
invalid_times = comparison[comparison['departure_home_to_work'] > comparison['departure_from_work']]

print(f"Number of agents with home-to-work departure > work-to-home departure: {len(invalid_times)}")
print(f"Percentage of total: {len(invalid_times) / len(workers) * 100:.2f}%")
print("\nSample of invalid cases:")
print(invalid_times.head(10))

# MISSING WORK ADDRESS

In [ ]:
# Agents with missing work coordinates (either work_x or work_y is null)
missing_work_coords = workers[workers[["work_x", "work_y"]].isna().any(axis=1)].copy()

# Counts
n_rows_missing = len(missing_work_coords)
n_agents_missing = missing_work_coords["participant_id"].nunique()

print(f"Rows with missing work_x or work_y: {n_rows_missing}")
print(f"Unique agents with missing work_x or work_y: {n_agents_missing}")

# Optional: inspect a sample
print(missing_work_coords[["participant_id", "work_x", "work_y"]].head(10))

# CAR AND COMPANY CAR STATS

In [ ]:
print(workers['has_car'].value_counts(dropna=False)/len(workers)*100)

In [ ]:
print(workers['has_company_car'].value_counts(dropna=False)/len(workers)*100)

# Teleworkers modal split

In [ ]:
full_pop = pd.read_csv('output/commuters_with_telework_tag.csv')

In [ ]:
tw = full_pop[full_pop["telework_today"] == 1].copy()

print(f"Total number of workers: {len(full_pop)}")
print(f"Total number of teleworkers: {len(tw)}")

In [ ]:
MODES = ["car", "pt", "bike", "walk"]

print("\nModal split (all):")
mc = full_pop["mode"].value_counts()
for m in MODES:
    n = mc.get(m, 0)
    print(f"{m:<6} {n/len(full_pop)*100:.1f}%")

print("\nModal split (teleworkers):")
mc = tw["mode"].value_counts()
for m in MODES:
    n = mc.get(m, 0)
    print(f"{m:<6} {n/len(tw)*100:.1f}%")